# Derive numeric version

`Model4DSTEM`is developed strictly for and with SymPy. It will work with basic types as well in many cases since SymPy allows this. However, it doesn not work e.g. with NumPy arrays. This example shows how that can be accomplished.

In [1]:
import numpy as np
import jax
import jax.numpy as jnp

from microscope_calibration.common.model import Model4DSTEM, PixelYX

In [2]:
default_model = Model4DSTEM.default()

In [3]:
default_model.trace(
    scan_pos=PixelYX(y=1, x=2),
    source_dy=0.01,
    source_dx=0.02,
)["detector"]

ResultSection(component=Plane(z=1.0), ray=Ray(x=0.0200000000000000, y=0.0100000000000000, dx=0.0200000000000000, dy=0.0100000000000000, z=1.0, pathlength=1.0, _one=1.0), sampling={'detector_px': PixelYX(y=200.000000000000, x=400.000000000000)})

In [4]:
default_model.trace(
    scan_pos=PixelYX(y=1, x=2),
    source_dy=0.01,
    source_dx=np.array((0.02, 0.03, 0.04, 0.05)),
)["detector"]

AttributeError: 'ImmutableDenseNDimArray' object has no attribute 'equals'

First, we create the type for the model. The argument is, essentially, the `modules` argument of SymPy's [`lambdify()`](https://docs.sympy.org/latest/modules/utilities/lambdify.html#sympy.utilities.lambdify.lambdify)

In [5]:
NP_Model4DSTEM = Model4DSTEM.lambdify(np)

In [6]:
np_model = NP_Model4DSTEM.from_model(default_model)

In [7]:
np_model.trace(
    scan_pos=PixelYX(y=1, x=2),
    source_dy=0.01,
    source_dx=np.array((0.02, 0.03, 0.04, 0.05)),
)["detector"]

ResultSection(component=Plane(z=1.0), ray=Ray(x=array([0.02, 0.03, 0.04, 0.05]), y=np.float64(0.01), dx=array([0.02, 0.03, 0.04, 0.05]), dy=np.float64(0.01), z=1.0, pathlength=1.0, _one=1.0), sampling={'detector_px': PixelYX(y=array([200., 200., 200., 200.]), x=array([ 400.,  600.,  800., 1000.]))})

## Broadcasting

Generally, broadcasting should work between the arguments like expected for NumPy expressions. However, this hasn't been tested in-depth yet. Feedback welcome!

In [8]:
y, x, dy, dx = np.ogrid[:11, :12, :13, :14]

In [9]:
four_dimensional = np_model.trace(
    scan_pos=PixelYX(y=y, x=x),
    source_dy=dy * 0.01,
    source_dx=dx * 0.01,
)["detector"]

In [10]:
four_dimensional.sampling["detector_px"].x

array([[[[   0.,  200.,  400., ..., 2200., 2400., 2600.],
         [   0.,  200.,  400., ..., 2200., 2400., 2600.],
         [   0.,  200.,  400., ..., 2200., 2400., 2600.],
         ...,
         [   0.,  200.,  400., ..., 2200., 2400., 2600.],
         [   0.,  200.,  400., ..., 2200., 2400., 2600.],
         [   0.,  200.,  400., ..., 2200., 2400., 2600.]],

        [[   0.,  200.,  400., ..., 2200., 2400., 2600.],
         [   0.,  200.,  400., ..., 2200., 2400., 2600.],
         [   0.,  200.,  400., ..., 2200., 2400., 2600.],
         ...,
         [   0.,  200.,  400., ..., 2200., 2400., 2600.],
         [   0.,  200.,  400., ..., 2200., 2400., 2600.],
         [   0.,  200.,  400., ..., 2200., 2400., 2600.]],

        [[   0.,  200.,  400., ..., 2200., 2400., 2600.],
         [   0.,  200.,  400., ..., 2200., 2400., 2600.],
         [   0.,  200.,  400., ..., 2200., 2400., 2600.],
         ...,
         [   0.,  200.,  400., ..., 2200., 2400., 2600.],
         [   0.,  200.,  4

All methods should work normally with the lambdified class

In [11]:
new_model = np_model.adjust_camera_length(23)
new_model

Model4DSTEM.lambdify(numpy)(overfocus=0.0, scan_pixel_pitch=1e-06, scan_center=PixelYX(y=0.0, x=0.0), scan_rotation=0.0, camera_length=23, detector_pixel_pitch=5e-05, detector_center=PixelYX(y=0.0, x=0.0), semiconv=0.001, flip_factor=1.0, descan_error=DescanError(pxo_pxi=0.0, pxo_pyi=0.0, pyo_pxi=0.0, pyo_pyi=0.0, sxo_pxi=0.0, sxo_pyi=0.0, syo_pxi=0.0, syo_pyi=0.0, offpxi=0.0, offpyi=0.0, offsxi=0.0, offsyi=0.0), detector_rotation=0.0)

The same approach enables differentiation with JAX. The default SymPy model is incompatible with gradients:

In [12]:
jax.jacobian(default_model.trace, argnums=(0, 1, 2))(
    PixelYX(y=1.0, x=2.0),  # scan_pos
    0.02,  # source_dx
    0.01,  # source_dy
)["detector"]

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


ConcretizationTypeError: Abstract tracer value encountered where concrete value is expected: traced array with shape float64[]
The problem arose with the `float` function. If trying to convert the data type of a value, try using `x.astype(float)` or `jnp.array(x, float)` instead.

See https://docs.jax.dev/en/latest/errors.html#jax.errors.ConcretizationTypeError

...but the JAX-lambdified version works without problems:

In [13]:
JAX_Model4DSTEM = Model4DSTEM.lambdify(jnp)

In [14]:
jax_model = JAX_Model4DSTEM.from_model(default_model)

In [15]:
jax.jacobian(jax_model.trace, argnums=(0, 1, 2))(
    PixelYX(y=1.0, x=2.0),  # scan_pos
    0.02,  # source_dx
    0.01,  # source_dy
)["detector"]

ResultSection(component=Plane(z=(PixelYX(y=Array(0., dtype=float64, weak_type=True), x=Array(0., dtype=float64, weak_type=True)), Array(0., dtype=float64, weak_type=True), Array(0., dtype=float64, weak_type=True))), ray=Ray(x=(PixelYX(y=Array(0., dtype=float64, weak_type=True), x=Array(0., dtype=float64, weak_type=True)), Array(1., dtype=float64, weak_type=True), Array(0., dtype=float64, weak_type=True)), y=(PixelYX(y=Array(0., dtype=float64, weak_type=True), x=Array(0., dtype=float64, weak_type=True)), Array(0., dtype=float64, weak_type=True), Array(1., dtype=float64, weak_type=True)), dx=(PixelYX(y=Array(0., dtype=float64, weak_type=True), x=Array(0., dtype=float64, weak_type=True)), Array(1., dtype=float64, weak_type=True), Array(0., dtype=float64, weak_type=True)), dy=(PixelYX(y=Array(0., dtype=float64, weak_type=True), x=Array(0., dtype=float64, weak_type=True)), Array(0., dtype=float64, weak_type=True), Array(1., dtype=float64, weak_type=True)), z=(PixelYX(y=Array(0., dtype=float

In real calculations one may want to define custom functions to calculate, solve, differentiate and optimize instead of working with e.g. trace() directly. In particular in combination with `jax.jit` that should be a lot more efficient since it can skip many calculations, and gives more flexibility over what is constant and what is variable. See model.ipynb for a simple example of that approach.